In [0]:
WITH date_bounds AS (
    SELECT
        MIN(valid_from) AS earliest,
        MAX(valid_to)   AS latest
    FROM osv.silver.v_pypi_history
    WHERE valid_to != '9999-12-31'
),
changing_keys AS (
    SELECT DISTINCT id
    FROM osv.silver.v_pypi_history
    WHERE valid_to != '9999-12-31'
)
SELECT
    v.valid_from,
    v.valid_to,
    v.id,
    v.summary,
    v.modified,
    v.withdrawn,
    v.current_flag
FROM osv.silver.v_pypi_history v
WHERE v.id IN (SELECT id FROM changing_keys)
AND v.valid_from BETWEEN (SELECT earliest FROM date_bounds) AND (SELECT latest FROM date_bounds)
ORDER BY v.id, v.valid_from;

In [0]:
%python

dbutils.widgets.text('vuln_id', '')
dbutils.widgets.dropdown('ecosystem', 'pypi', ['pypi', 'cran'])

vuln_id = dbutils.widgets.get('vuln_id')
ecosystem = dbutils.widgets.get('ecosystem')

view_name = f'osv.silver.v_{ecosystem}_history'

latest_valid = spark.sql(f"SELECT MAX(valid_to) AS latest FROM {view_name} WHERE valid_to != '9999-12-31'").collect()[0]['latest']

query = f"""
SELECT
    valid_from,
    CASE
        WHEN valid_to = '9999-12-31' THEN DATE('{latest_valid}')
        ELSE valid_to
    END AS valid_to,
    summary,
    withdrawn,
    modified
FROM {view_name}
WHERE id = '{vuln_id}'
ORDER BY valid_from
"""

df = spark.sql(query).toPandas()

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df['valid_from'] = pd.to_datetime(df['valid_from'])
df['valid_to'] = pd.to_datetime(df['valid_to'])

fig, ax = plt.subplots(figsize=(12, max(len(df) * 0.8, 2)))

for i, row in df.iterrows():
    color = 'salmon' if pd.notna(row['withdrawn']) and row['withdrawn'] else 'steelblue'
    ax.barh(
        y=i,
        width=(row['valid_to'] - row['valid_from']).days,
        left=row['valid_from'],
        height=0.8,
        color=color
    )

labels = []
for _, row in df.iterrows():
    w = ' [WITHDRAWN]' if pd.notna(row['withdrawn']) and row['withdrawn'] else ''
    summary = (str(row['summary']) if pd.notna(row['summary']) else '')[:60]
    labels.append(f'{summary}{w}')

ax.set_yticks(range(len(df)))
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlim(left=df['valid_from'].min(), right=pd.Timestamp(str(latest_valid)) + pd.Timedelta(days=1))
ax.set_xlabel('Date')
ax.set_title(f'{vuln_id} ({ecosystem})')

ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
fig.autofmt_xdate(rotation=0)

plt.tight_layout()
plt.show()

In [0]:
SELECT
    id,
    COUNT(*) AS version_count,
    MIN(valid_from) AS first_seen,
    MAX(CASE WHEN current_flag = 'Yes' THEN valid_from END) AS last_changed,
    MAX(CASE WHEN current_flag = 'Yes' THEN withdrawn END) AS current_withdrawn_date
FROM osv.silver.v_pypi_history
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY version_count DESC
LIMIT 25;